In [10]:

"""
=============================================================
  SPORTS MERCHANDISE — COMPLETE MARKET BASKET ANALYSIS
  Dataset: dataset_2_sports_merch_market_basket.xlsx
=============================================================
Sections:
  1.  Data Loading & Quality Check
  2.  Exploratory Data Analysis (EDA)
  3.  Revenue & Sales Performance
  4.  Fan Segment & Channel Analysis
  5.  Match Context & Campaign Analysis
  6.  Basket Size & Returns Analysis
  7.  Market Basket Analysis (Apriori + Association Rules)
  8.  Product Co-occurrence Heatmap
  9.  Temporal Trends
  10. Strategic Interpretation Summary
=============================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings("ignore")
from itertools import combinations
from collections import defaultdict, Counter
import os

# ── colour palette ────────────────────────────────────────────────────
CLUB_BLUE   = "#1B3A6B"
CLUB_GOLD   = "#C9A84C"
CLUB_RED    = "#C1121F"
PALE_BG     = "#F5F7FA"
GREY        = "#6C757D"

PALETTE = [CLUB_BLUE, CLUB_GOLD, CLUB_RED, "#2ECC71", "#9B59B6",
           "#E67E22", "#1ABC9C", "#E74C3C", "#3498DB", "#F39C12"]

plt.rcParams.update({
    "figure.facecolor": PALE_BG,
    "axes.facecolor":   PALE_BG,
    "axes.edgecolor":   GREY,
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "font.family":      "DejaVu Sans",
    "font.size":        10,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "legend.fontsize":  9,
})

OUTPUT = "."  # saves figures in your current working directory
os.makedirs(OUTPUT, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════
# 1. DATA LOADING & QUALITY CHECK
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  SECTION 1 — DATA LOADING & QUALITY CHECK")
print("="*70)

df = pd.read_excel("/content/dataset_2_sports_merch_market_basket (1).xlsx")
df["invoice_date"] = pd.to_datetime(df["invoice_date"])

print(f"  Total rows        : {len(df):,}")
print(f"  Total columns     : {df.shape[1]}")
print(f"  Unique invoices   : {df['invoice_id'].nunique():,}")
print(f"  Unique customers  : {df['customer_id'].nunique():,}")
print(f"  Unique products   : {df['product_id'].nunique():,}")
print(f"  Date range        : {df['invoice_date'].min().date()} → {df['invoice_date'].max().date()}")

print("\n  Missing values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  → None detected")

print("\n  Duplicate rows:", df.duplicated().sum())

print("\n  Returned transactions :", df[df["returned_flag"]==1]["invoice_id"].nunique())
print("  Return rate (lines)   :", f"{df['returned_flag'].mean()*100:.2f}%")

print("\n  Revenue summary (INR):")
rev = df["line_revenue_inr"]
print(f"  Total revenue   : ₹{rev.sum():,.0f}")
print(f"  Avg per line    : ₹{rev.mean():,.2f}")
print(f"  Median per line : ₹{rev.median():,.2f}")
print(f"  Max per line    : ₹{rev.max():,.2f}")

# ═══════════════════════════════════════════════════════════════════════
# 2. EXPLORATORY DATA ANALYSIS — Fig 1
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  SECTION 2 — EDA OVERVIEW  (Figure 1)")
print("="*70)

fig1, axes = plt.subplots(2, 3, figsize=(18, 11))
fig1.suptitle("Sports Merchandise — EDA Overview", fontsize=16,
              fontweight="bold", color=CLUB_BLUE, y=1.01)

# 2a — Category revenue
ax = axes[0, 0]
cat_rev = df.groupby("category")["line_revenue_inr"].sum().sort_values(ascending=False)
bars = ax.bar(cat_rev.index, cat_rev.values / 1e6,
              color=[PALETTE[i % len(PALETTE)] for i in range(len(cat_rev))])
ax.set_title("Revenue by Category (₹ Mn)")
ax.set_xlabel("Category")
ax.set_ylabel("Revenue (₹ Mn)")
ax.tick_params(axis="x", rotation=30)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2,
            f"₹{b.get_height():.1f}", ha="center", va="bottom", fontsize=8)

# 2b — Top 10 products by revenue
ax = axes[0, 1]
prod_rev = df.groupby("product_name")["line_revenue_inr"].sum().sort_values(ascending=False).head(10)
ax.barh(prod_rev.index[::-1], prod_rev.values[::-1] / 1e6, color=CLUB_BLUE)
ax.set_title("Top 10 Products by Revenue (₹ Mn)")
ax.set_xlabel("Revenue (₹ Mn)")
for i, v in enumerate(prod_rev.values[::-1]):
    ax.text(v/1e6 + 0.05, i, f"₹{v/1e6:.1f}M", va="center", fontsize=8)

# 2c — Discount distribution
ax = axes[0, 2]
disc_counts = df["discount_pct"].value_counts().sort_index()
ax.bar(disc_counts.index.astype(str), disc_counts.values, color=CLUB_GOLD)
ax.set_title("Distribution of Discount %")
ax.set_xlabel("Discount %")
ax.set_ylabel("Number of Line Items")

# 2d — Sales channel share (donut)
ax = axes[1, 0]
ch = df.groupby("sales_channel")["line_revenue_inr"].sum()
wedges, texts, autotexts = ax.pie(
    ch.values, labels=ch.index, autopct="%1.1f%%",
    colors=PALETTE[:len(ch)], startangle=140,
    wedgeprops=dict(width=0.6))
ax.set_title("Revenue by Sales Channel")

# 2e — Fan segment revenue
ax = axes[1, 1]
seg_rev = df.groupby("fan_segment")["line_revenue_inr"].sum().sort_values(ascending=False)
bars = ax.bar(seg_rev.index, seg_rev.values / 1e6,
              color=[PALETTE[i % len(PALETTE)] for i in range(len(seg_rev))])
ax.set_title("Revenue by Fan Segment (₹ Mn)")
ax.set_xlabel("Fan Segment")
ax.set_ylabel("Revenue (₹ Mn)")
ax.tick_params(axis="x", rotation=25)

# 2f — Units sold by subcategory (top 12)
ax = axes[1, 2]
sub_units = df.groupby("subcategory")["quantity"].sum().sort_values(ascending=False).head(12)
ax.barh(sub_units.index[::-1], sub_units.values[::-1], color=CLUB_RED, alpha=0.85)
ax.set_title("Units Sold — Top 12 Subcategories")
ax.set_xlabel("Total Units")

plt.tight_layout()
fig1.savefig(f"{OUTPUT}/fig1_eda_overview.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → fig1_eda_overview.png saved")

# ═══════════════════════════════════════════════════════════════════════
# 3. REVENUE & SALES PERFORMANCE — Fig 2
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  SECTION 3 — REVENUE & SALES PERFORMANCE  (Figure 2)")
print("="*70)

# Invoice-level aggregates
inv_df = df.groupby("invoice_id").agg(
    basket_revenue=("line_revenue_inr", "sum"),
    basket_items=("product_id", "count"),
    channel=("sales_channel", "first"),
    segment=("fan_segment", "first"),
    match_ctx=("match_context", "first"),
    date=("invoice_date", "first"),
    returned=("returned_flag", "max"),
    discount_pct=("discount_pct", "mean")
).reset_index()

print(f"  Avg basket value : ₹{inv_df['basket_revenue'].mean():,.0f}")
print(f"  Median basket    : ₹{inv_df['basket_revenue'].median():,.0f}")
print(f"  Avg items/basket : {inv_df['basket_items'].mean():.2f}")

fig2, axes = plt.subplots(2, 3, figsize=(18, 11))
fig2.suptitle("Revenue & Sales Performance", fontsize=16,
              fontweight="bold", color=CLUB_BLUE, y=1.01)

# 3a — Basket value distribution
ax = axes[0, 0]
ax.hist(inv_df["basket_revenue"].clip(upper=inv_df["basket_revenue"].quantile(0.99)),
        bins=60, color=CLUB_BLUE, edgecolor="white", linewidth=0.4, alpha=0.85)
ax.axvline(inv_df["basket_revenue"].mean(), color=CLUB_RED, lw=2, linestyle="--",
           label=f"Mean ₹{inv_df['basket_revenue'].mean():,.0f}")
ax.axvline(inv_df["basket_revenue"].median(), color=CLUB_GOLD, lw=2, linestyle="--",
           label=f"Median ₹{inv_df['basket_revenue'].median():,.0f}")
ax.set_title("Basket Value Distribution")
ax.set_xlabel("Basket Revenue (₹)")
ax.set_ylabel("Number of Invoices")
ax.legend()

# 3b — Monthly revenue trend
df["month"] = df["invoice_date"].dt.to_period("M")
monthly = df.groupby("month")["line_revenue_inr"].sum() / 1e6
ax = axes[0, 1]
ax.fill_between(range(len(monthly)), monthly.values, alpha=0.3, color=CLUB_BLUE)
ax.plot(range(len(monthly)), monthly.values, color=CLUB_BLUE, lw=2.5, marker="o")
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels([str(p) for p in monthly.index], rotation=30, fontsize=8)
ax.set_title("Monthly Revenue Trend (₹ Mn)")
ax.set_ylabel("Revenue (₹ Mn)")

# 3c — Revenue by match context
ax = axes[0, 2]
mc_rev = df.groupby("match_context")["line_revenue_inr"].sum().sort_values(ascending=False)
bars = ax.bar(mc_rev.index, mc_rev.values / 1e6, color=PALETTE[:len(mc_rev)])
ax.set_title("Revenue by Match Context (₹ Mn)")
ax.tick_params(axis="x", rotation=30)
ax.set_ylabel("Revenue (₹ Mn)")

# 3d — AOV by channel x segment heatmap
ax = axes[1, 0]
pivot = inv_df.pivot_table(values="basket_revenue",
                            index="segment", columns="channel", aggfunc="mean")
cmap = LinearSegmentedColormap.from_list("club", ["#E8F4FD", CLUB_BLUE])
im = ax.imshow(pivot.values, aspect="auto", cmap=cmap)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title("Avg Basket Value: Segment × Channel (₹)")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"₹{val:,.0f}", ha="center", va="center",
                    fontsize=7.5, color="white" if val > pivot.values.max()*0.6 else CLUB_BLUE)
plt.colorbar(im, ax=ax, shrink=0.85)

# 3e — Campaign source revenue
ax = axes[1, 1]
camp = df.groupby("campaign_source")["line_revenue_inr"].sum().sort_values(ascending=False)
ax.bar(camp.index, camp.values / 1e6, color=[PALETTE[i % len(PALETTE)] for i in range(len(camp))])
ax.set_title("Revenue by Campaign Source (₹ Mn)")
ax.tick_params(axis="x", rotation=30)
ax.set_ylabel("Revenue (₹ Mn)")

# 3f — Price tier analysis
ax = axes[1, 2]
df["price_tier"] = pd.cut(df["unit_price_inr"],
                           bins=[0, 499, 999, 1999, 3999, 99999],
                           labels=["<₹500", "₹500-999", "₹1K-2K", "₹2K-4K", "₹4K+"])
tier_rev = df.groupby("price_tier", observed=True)["line_revenue_inr"].sum()
ax.pie(tier_rev.values, labels=tier_rev.index, autopct="%1.1f%%",
       colors=PALETTE[:len(tier_rev)], startangle=140,
       wedgeprops=dict(width=0.6))
ax.set_title("Revenue Share by Price Tier")

plt.tight_layout()
fig2.savefig(f"{OUTPUT}/fig2_revenue_sales.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → fig2_revenue_sales.png saved")

# ═══════════════════════════════════════════════════════════════════════
# 4. FAN SEGMENT DEEP-DIVE — Fig 3
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  SECTION 4 — FAN SEGMENT ANALYSIS  (Figure 3)")
print("="*70)

fig3, axes = plt.subplots(2, 2, figsize=(16, 11))
fig3.suptitle("Fan Segment Deep-Dive", fontsize=16,
              fontweight="bold", color=CLUB_BLUE, y=1.01)

segments = df["fan_segment"].unique()

# 4a — Avg basket by segment
ax = axes[0, 0]
seg_basket = inv_df.groupby("segment")["basket_revenue"].mean().sort_values(ascending=False)
bars = ax.bar(seg_basket.index, seg_basket.values, color=PALETTE[:len(seg_basket)])
ax.set_title("Avg Basket Value by Fan Segment (₹)")
ax.set_ylabel("Avg Basket (₹)")
ax.tick_params(axis="x", rotation=25)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 50,
            f"₹{b.get_height():,.0f}", ha="center", va="bottom", fontsize=9)

# 4b — Top category per segment (stacked bar)
ax = axes[0, 1]
seg_cat = df.groupby(["fan_segment", "category"])["line_revenue_inr"].sum().unstack(fill_value=0)
seg_cat_pct = seg_cat.div(seg_cat.sum(axis=1), axis=0) * 100
seg_cat_pct.plot(kind="bar", stacked=True, ax=ax, color=PALETTE[:len(seg_cat_pct.columns)],
                 edgecolor="white", linewidth=0.4)
ax.set_title("Category Mix by Fan Segment (%)")
ax.set_ylabel("Revenue Share (%)")
ax.tick_params(axis="x", rotation=30)
ax.legend(loc="upper right", fontsize=8, ncol=2)

# 4c — Return rate by segment
ax = axes[1, 0]
ret_seg = df.groupby("fan_segment")["returned_flag"].mean() * 100
bars = ax.bar(ret_seg.index, ret_seg.values, color=[CLUB_RED if v > ret_seg.mean() else CLUB_BLUE
                                                      for v in ret_seg.values])
ax.axhline(ret_seg.mean(), color=CLUB_GOLD, lw=1.5, linestyle="--",
           label=f"Avg {ret_seg.mean():.2f}%")
ax.set_title("Return Rate by Fan Segment (%)")
ax.set_ylabel("Return Rate (%)")
ax.tick_params(axis="x", rotation=25)
ax.legend()

# 4d — Customer count and revenue volume comparison
ax = axes[1, 1]
seg_stats = df.groupby("fan_segment").agg(
    customers=("customer_id", "nunique"),
    revenue=("line_revenue_inr", "sum")
).reset_index()
x = np.arange(len(seg_stats))
w = 0.4
ax2 = ax.twinx()
ax.bar(x - w/2, seg_stats["customers"], width=w, color=CLUB_BLUE, label="Customers")
ax2.bar(x + w/2, seg_stats["revenue"] / 1e6, width=w, color=CLUB_GOLD, label="Revenue (₹Mn)")
ax.set_xticks(x)
ax.set_xticklabels(seg_stats["fan_segment"], rotation=30, fontsize=9)
ax.set_ylabel("Unique Customers", color=CLUB_BLUE)
ax2.set_ylabel("Revenue (₹ Mn)", color="#8B6914")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)

plt.tight_layout()
fig3.savefig(f"{OUTPUT}/fig3_fan_segments.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → fig3_fan_segments.png saved")

# ═══════════════════════════════════════════════════════════════════════
# 5. MATCH CONTEXT & TEMPORAL — Fig 4
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  SECTION 5 — MATCH CONTEXT & TEMPORAL  (Figure 4)")
print("="*70)

fig4, axes = plt.subplots(2, 2, figsize=(16, 11))
fig4.suptitle("Match Context & Temporal Behaviour", fontsize=16,
              fontweight="bold", color=CLUB_BLUE, y=1.01)

# 5a — Revenue by match context + channel (grouped)
ax = axes[0, 0]
mc_ch = df.groupby(["match_context", "sales_channel"])["line_revenue_inr"].sum().unstack(fill_value=0)
# Ensure no NaN values from division by zero when calculating percentages
row_sums_mc_ch = mc_ch.sum(axis=1)
mc_ch_pct = mc_ch.div(row_sums_mc_ch.replace(0, np.nan), axis=0) * 100
mc_ch_pct = mc_ch_pct.fillna(0)
mc_ch_pct.plot(kind="bar", ax=ax, color=PALETTE[:len(mc_ch_pct.columns)],
               edgecolor="white", linewidth=0.4)
ax.set_title("Channel Mix by Match Context (%)")
ax.set_ylabel("Revenue Share (%)")
ax.tick_params(axis="x", rotation=30)
ax.legend(fontsize=8, ncol=2)

# 5b — Category preference by match context
ax = axes[0, 1]
mc_cat = df.groupby(["match_context", "category"])["line_revenue_inr"].sum().unstack(fill_value=0)
mc_cat.plot(kind="bar", stacked=True, ax=ax, color=PALETTE[:len(mc_cat.columns)],
            edgecolor="white", linewidth=0.4)
ax.set_title("Category Mix by Match Context (₹)")
ax.set_ylabel("Revenue (₹)")
ax.tick_params(axis="x", rotation=30)
ax.legend(loc="upper right", fontsize=8)

# 5c — Weekly day revenue pattern
ax = axes[1, 0]
df["weekday"] = df["invoice_date"].dt.day_name()
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
day_rev = df.groupby("weekday")["line_revenue_inr"].sum().reindex(day_order) / 1e6
ax.bar(day_rev.index, day_rev.values, color=[CLUB_GOLD if d in ["Saturday","Sunday"] else CLUB_BLUE
                                              for d in day_rev.index])
ax.set_title("Revenue by Day of Week (₹ Mn)")
ax.set_ylabel("Revenue (₹ Mn)")
ax.tick_params(axis="x", rotation=30)

# 5d — Basket size over time (weekly)
ax = axes[1, 1]
df["week"] = df["invoice_date"].dt.to_period("W")
weekly_basket = inv_df.copy()
weekly_basket["week"] = weekly_basket["date"].dt.to_period("W")
wb = weekly_basket.groupby("week")[


  SECTION 1 — DATA LOADING & QUALITY CHECK
  Total rows        : 77,041
  Total columns     : 18
  Unique invoices   : 22,000
  Unique customers  : 5,119
  Unique products   : 50
  Date range        : 2026-01-01 → 2026-05-31

  Missing values:
campaign_source    29170
dtype: int64

  Duplicate rows: 0

  Returned transactions : 1361
  Return rate (lines)   : 1.81%

  Revenue summary (INR):
  Total revenue   : ₹86,749,471
  Avg per line    : ₹1,126.02
  Median per line : ₹699.00
  Max per line    : ₹5,697.00

  SECTION 2 — EDA OVERVIEW  (Figure 1)
  → fig1_eda_overview.png saved

  SECTION 3 — REVENUE & SALES PERFORMANCE  (Figure 2)
  Avg basket value : ₹3,943
  Median basket    : ₹3,497
  Avg items/basket : 3.50
  → fig2_revenue_sales.png saved

  SECTION 4 — FAN SEGMENT ANALYSIS  (Figure 3)
  → fig3_fan_segments.png saved

  SECTION 5 — MATCH CONTEXT & TEMPORAL  (Figure 4)
  → fig4_match_temporal.png saved

  SECTION 6 — BASKET SIZE & RETURNS  (Figure 5)
  → fig5_basket_returns.png s